# ROI G. Biv — Consensus Cell Detection Pipeline
### Google Colab · End-to-End Execution

**ROI G. Biv** is a consensus cell-detection pipeline for two-photon calcium imaging data.
It combines two independent detectors — Suite2p and a fine-tuned Cellpose model — and
merges their outputs using spatial overlap scoring. The result is a three-tier confidence
system that is more reliable than either tool alone:

| Tier | Condition | Interpretation |
|------|-----------|----------------|
| **GOLD** | Both Suite2p passes agree (IoU ≥ 0.3) | Highest confidence — morphology and activity agree |
| **SILVER** | Anatomy pass only | Neuron-shaped but silent during this recording |
| **BRONZE** | Activity pass only | Was active; not anatomically prominent in mean image |

**This notebook runs the complete pipeline in Colab.** The only input required is your
pre-motion-corrected TIF stacks uploaded to Google Drive.

```
Raw TIF stacks (pre-motion-corrected)
        │
        ├──► Stage 1: Suite2p activity pass   (anatomical_only=0)
        │            temporal pixel correlation → ROI candidates
        │
        ├──► Stage 2: Suite2p anatomy pass    (anatomical_only=1)
        │            mean-image morphology   → silent cell candidates
        │
        ├──► Stage 3: Extract projections
        │            ops.npy → mean image + Vcorr map (TIF)
        │
        │    Stage 4: (pre-trained model shipped with this release)
        │
        └──► Stage 5 + 6: Union ROI building + consensus scoring
                     Hungarian IoU matching → GOLD / SILVER / BRONZE
                     Cellpose probability heatmap per ROI
```

> **Requirements:** GPU runtime (Runtime → Change runtime type → T4 GPU).
> All cells must be run top-to-bottom on first use. Subsequent runs skip
> already-processed FOVs automatically.

## Step 0 — Install Dependencies

Suite2p is installed first so its dependency pins settle before Cellpose is
upgraded. This avoids the conflict where Suite2p's `anatomical_only=2` mode
imports Cellpose 3.x internally — since this pipeline never calls that code
path, both coexist safely after the upgrade.

The `roigbiv` package is then installed from the latest GitHub release.
It provides all pipeline logic; no local files need to be uploaded.

> **One-time step.** Re-running this cell in an existing runtime is safe but
> unnecessary if the packages are already installed.

In [ ]:
# roigbiv release version — update this when a new release is published.
# The wheel installs directly without a build step (fast, no backend issues).
ROIGBIV_VERSION = "0.1.0"

# 1. Suite2p (lets its dependency pins settle first)
!pip install suite2p -q

# 2. Cellpose 4.x (Suite2p may have pinned an older version)
!pip install "cellpose==4.0.9" --upgrade -q

# 3. roigbiv pre-built wheel from GitHub release
#    Replace YOUR_GITHUB_USERNAME with the repository owner before running.
!pip install "https://github.com/YOUR_GITHUB_USERNAME/roigbiv/releases/download/v{ROIGBIV_VERSION}/roigbiv-{ROIGBIV_VERSION}-py3-none-any.whl" -q

print("Installation complete.")

## Step 1 — Configuration

**Edit the cell below.** All other cells can be run without modification.

| Parameter | Default | Description |
|-----------|---------|-------------|
| `DRIVE_ROOT` | — | Path to the Google Drive folder containing your TIF files |
| `FRAME_RATE` | `30.0` | Acquisition frame rate in Hz |
| `TAU` | `1.0` | GCaMP decay constant in seconds (GCaMP6s ≈ 1.0 s; GCaMP7f ≈ 0.7 s) |
| `DIAMETER` | `30` | Expected soma diameter in pixels; Cellpose rescales to this size internally |
| `USE_VCORR` | `True` | Stack Vcorr map as 2nd input channel alongside the mean projection; provides temporal activity signature and generally improves segmentation |
| `IOU_THRESHOLD` | `0.3` | Minimum spatial overlap (IoU) between anatomy and activity ROIs to be classified GOLD; calibrated for Suite2p pixelated footprints vs. smoother contours |
| `OUTPUT_TIERS` | `["gold", "silver"]` | Tiers to include in the default output mask |
| `DO_REGISTRATION` | `False` | Set `True` if TIF files are **not** pre-motion-corrected |
| `MODEL_RELEASE_URL` | — | Direct URL to the deployed Cellpose checkpoint (GitHub release asset) |

In [ ]:
# ── USER CONFIGURATION ── Edit this cell before running the pipeline ────────

# Path to the Google Drive folder containing your TIF stacks.
# Can be a flat directory, nested subdirectories, or a .tar.gz / .zip archive.
DRIVE_ROOT = "/content/drive/MyDrive/calcium_imaging"

# Acquisition frame rate in Hz. Check your microscope acquisition log.
FRAME_RATE = 30.0

# GCaMP decay time constant in seconds.
# GCaMP6s ≈ 1.0 s  |  GCaMP6f ≈ 0.4 s  |  GCaMP7f ≈ 0.7 s
TAU = 1.0

# Expected soma diameter in pixels (used by Cellpose for internal rescaling).
DIAMETER = 30

# Stack the Vcorr temporal correlation map as a second input channel.
# Provides functional activity information alongside morphology.
USE_VCORR = True

# Minimum IoU for GOLD-tier classification (0.3 = calibrated default).
IOU_THRESHOLD = 0.3

# Tiers to include in the default output mask.
# "gold" + "silver" captures active neurons and silent neurons.
# Add "bronze" to include activity-only detections.
OUTPUT_TIERS = ["gold", "silver"]

# Set True if TIF files are NOT motion-corrected.
# False = skip registration (stacks are pre-corrected, e.g. *_mc.tif).
DO_REGISTRATION = False

# Pre-trained Cellpose checkpoint distributed with this release.
# Replace YOUR_GITHUB_USERNAME with the repository owner.
MODEL_RELEASE_URL = (
    "https://github.com/YOUR_GITHUB_USERNAME/roigbiv"
    "/releases/latest/download/current_model"
)

# ── Internal paths (no need to edit) ────────────────────────────────────────
S2P_ACTIVITY_DIR  = "/content/s2p_activity"
S2P_ANATOMY_DIR   = "/content/s2p_anatomy"
PROJECTIONS_DIR   = "/content/projections"
OUTPUT_DIR        = "/content/roigbiv_output"
MODEL_CACHE_PATH  = "/content/roigbiv_model/current_model"

print("Configuration loaded.")
print(f"  Drive root:   {DRIVE_ROOT}")
print(f"  Frame rate:   {FRAME_RATE} Hz")
print(f"  Tau:          {TAU} s")
print(f"  Diameter:     {DIAMETER} px")
print(f"  Vcorr:        {USE_VCORR}")
print(f"  IoU threshold:{IOU_THRESHOLD}")
print(f"  Output tiers: {OUTPUT_TIERS}")
print(f"  Registration: {DO_REGISTRATION}")

## Step 2 — Mount Drive & Discover TIF Files

Google Drive is mounted at `/content/drive`. The pipeline then recursively
scans `DRIVE_ROOT` for TIF stacks.

**Supported input formats:**
- Single `.tif` / `.tiff` files (case-insensitive)
- Flat directories of TIF files
- Nested subdirectories
- `.tar.gz`, `.zip`, or `.tgz` archives — automatically extracted before scanning

Each file is validated to be a 3D stack (frames × height × width). Single-frame
images or unreadable files produce a clear error before any processing begins.

> **Expected filename convention:** pre-motion-corrected stacks end in `_mc.tif`.
> The `_mc` suffix is stripped when naming Suite2p outputs. Files without `_mc`
> are accepted as-is.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from roigbiv.io import discover_tifs, validate_tif
from pathlib import Path

tif_files = discover_tifs(DRIVE_ROOT)

if not tif_files:
    raise FileNotFoundError(
        f"No TIF files found under {DRIVE_ROOT}.\n"
        "Check that DRIVE_ROOT is set correctly and files have been uploaded."
    )

print(f"Found {len(tif_files)} TIF file(s):")
n_valid = 0
for tif in tif_files:
    try:
        stem, shape = validate_tif(tif)
        print(f"  {tif.name:60s}  {shape[0]} frames  {shape[1]}x{shape[2]} px")
        n_valid += 1
    except ValueError as e:
        print(f"  WARNING: {e}")

print(f"\n{n_valid}/{len(tif_files)} files valid and ready for processing.")

## Step 3 — Download Pre-trained Cellpose Model

The deployed model is a Cellpose `cyto3` checkpoint fine-tuned on manually
annotated neurons from this imaging setup (GCaMP6s, ventral striatum /
prefrontal cortex, ~30 Hz acquisition). It was trained on two-channel input:
channel 1 = mean projection, channel 2 = Vcorr temporal correlation map.

The checkpoint is a standard Cellpose ZIP-archive format and can also be
loaded directly in the Cellpose GUI (`CellposeModel(pretrained_model=path)`).

The download is skipped if the model is already cached from a previous run.

In [ ]:
from roigbiv.io import download_model

model_path = download_model(MODEL_RELEASE_URL, MODEL_CACHE_PATH)
print(f"\nModel ready: {model_path}")

## Step 4 — Suite2p Activity Pass

Suite2p detects ROIs by finding pixels that **co-vary over time** — groups
of pixels that tend to brighten and dim together. This is the standard
fluorescence-correlation-based detection mode (`anatomical_only=0`).

**What it computes:**
1. A pixel-level temporal correlation map (Vcorr) — bright islands = likely neurons
2. Spatially connected ROI candidates above a detection threshold
3. Neuropil subtraction (annulus model, `inner_neuropil_radius=2`)
4. OASIS spike deconvolution (decay constant = `TAU`)

**What it captures:** Neurons that were **active during the recording**.
Silent cells are missed — that is what the anatomy pass (Step 5) addresses.

**Resumability:** FOVs with existing `stat.npy` outputs are automatically skipped.
Re-running this cell after a Colab disconnect is safe.

`data.bin` (~500 MB per FOV) is deleted immediately after each FOV completes to
keep disk usage bounded.

In [ ]:
from roigbiv.suite2p import run_suite2p_batch

run_suite2p_batch(
    tif_files,
    output_dir   = S2P_ACTIVITY_DIR,
    fs           = FRAME_RATE,
    tau          = TAU,
    anatomical_only  = 0,        # activity-based detection
    do_registration  = DO_REGISTRATION,
)

## Step 5 — Suite2p Anatomy Pass + Extract Projections

### Anatomy pass (`anatomical_only=1`)

The anatomy pass runs the same Suite2p pipeline but detects ROIs from the
**time-averaged mean image alone**, using spatial gradient and morphology cues
rather than temporal correlations. It finds:
- **Silent neurons** that did not fire during the recording window
- Neurons with low SNR that the activity pass missed

This pass does **not** import Cellpose internally (`anatomical_only=1` uses
Suite2p's own morphological detector). The two packages coexist safely.

### Projection extraction

After Suite2p runs, the mean projection (`ops["meanImg"]`) and Vcorr map
(`ops["Vcorr"]`) are extracted from `ops.npy` and saved as TIF files.
These are required as input to the union ROI building step.

In [ ]:
from roigbiv.suite2p import run_suite2p_batch
from roigbiv.io import extract_projections

# Anatomy pass
run_suite2p_batch(
    tif_files,
    output_dir   = S2P_ANATOMY_DIR,
    fs           = FRAME_RATE,
    tau          = TAU,
    anatomical_only  = 1,        # mean-image morphology detection
    do_registration  = DO_REGISTRATION,
)

# Extract mean images and Vcorr maps from the activity-pass ops.npy
extract_projections(S2P_ACTIVITY_DIR, PROJECTIONS_DIR)

## Step 6 — Union ROI Building + Consensus Scoring

This is the core of the pipeline. All three candidate pools are merged and
scored in four sub-steps:

### 1. Merge Suite2p passes

Activity-pass ROIs and anatomy-pass ROIs are matched using
**pairwise IoU** (Intersection over Union) and the **Hungarian algorithm**
(`scipy.optimize.linear_sum_assignment`). The Hungarian algorithm finds the
globally optimal one-to-one pairing that maximises total IoU. Pairs with
IoU ≥ `IOU_THRESHOLD` become GOLD matches.

### 2. Tier assignment

| Tier | Condition | Mask source |
|------|-----------|-------------|
| GOLD | Activity **and** anatomy agree (IoU ≥ 0.3) | Anatomy boundary |
| SILVER | Anatomy only (no match or IoU < threshold) | Anatomy boundary |
| BRONZE | Activity only (no match or IoU < threshold) | Activity pixels |

GOLD and SILVER ROIs use anatomy boundaries because Suite2p's
mean-image pass tends to produce smoother, more anatomically accurate
contours than the temporal-correlation pass.

### 3. Cellpose probability scoring

The fine-tuned Cellpose model is run in **probability-extraction mode**
(`cellprob_threshold=-6` retains all candidates). This produces a per-pixel
cell-probability heatmap. Each union ROI is scored by the **mean Cellpose
probability over its pixel footprint**. This score enables post-hoc
thresholding independent of the tier system.

### 4. Output

| File | Contents |
|------|----------|
| `{stem}_all_s2p_masks.tif` | uint16 label image — each integer = one ROI |
| `{stem}_roi_cellprob.tif` | float32 Cellpose probability heatmap |
| `scored_rois_summary.csv` | Per-ROI: tier, IoU, Suite2p probs, Cellpose prob |

In [ ]:
from roigbiv.union import build_union_batch

summary = build_union_batch(
    tif_files       = tif_files,
    activity_dir    = S2P_ACTIVITY_DIR,
    anatomy_dir     = S2P_ANATOMY_DIR,
    projections_dir = PROJECTIONS_DIR,
    model_path      = model_path,
    output_dir      = OUTPUT_DIR,
    diameter        = DIAMETER,
    iou_threshold   = IOU_THRESHOLD,
    tiers           = OUTPUT_TIERS,
    use_vcorr       = USE_VCORR,
)

if not summary.empty:
    print("\nTier breakdown across all FOVs:")
    print(summary.groupby("tier")["roi_id"].count().rename("count"))
    print("\nFirst few rows:")
    display(summary.head(10))

## Step 7 — Interactive Viewer

The viewer below lets you explore processed FOVs directly in this notebook.

**Controls:**
- **FOV dropdown** — select which field of view to display
- **Tier selector** — toggle GOLD / SILVER / BRONZE visibility
- **Min prob slider** — hide ROIs below a minimum Cellpose probability
  (range −6 to +6; 0 = include most detections; 3+ = high confidence only)

**Color coding:**
- Gold contours = GOLD tier
- Cyan contours = SILVER tier
- Tomato contours = BRONZE tier

> The viewer renders ROI boundary contours using 1-pixel dilation.
> For large FOVs, rendering may take a few seconds per update.

In [ ]:
from roigbiv.viz import create_colab_viewer

create_colab_viewer(OUTPUT_DIR)

## Step 8 — Save Results to Drive

Results are copied from Colab's local `/content/` storage to your Google Drive.

**Output files per FOV:**

| File | Description |
|------|-------------|
| `{stem}_all_s2p_masks.tif` | uint16 ROI label image (all tiers combined) |
| `{stem}_roi_cellprob.tif` | float32 Cellpose probability per ROI pixel |
| `scored_rois_summary.csv` | Per-ROI table for downstream analysis |

**Using the results:**
- Load `{stem}_all_s2p_masks.tif` in Napari, FIJI, or Python (`tifffile.imread`)
  to inspect or further filter ROIs.
- Filter `scored_rois_summary.csv` by `tier` and `cellpose_mean_prob` to select
  your desired confidence level:
  ```python
  import pandas as pd
  df = pd.read_csv("scored_rois_summary.csv")
  high_conf = df[(df["tier"].isin(["GOLD", "SILVER"])) & (df["cellpose_mean_prob"] > 0.5)]
  ```
- Use ROI mask integer IDs to extract ΔF/F traces from raw stacks with
  `scripts/extract_traces.py` (local) or a custom extraction loop.

In [ ]:
import shutil
from pathlib import Path

dest = Path(DRIVE_ROOT) / "roigbiv_output"
shutil.copytree(OUTPUT_DIR, str(dest), dirs_exist_ok=True)

output_files = list(dest.glob("*"))
print(f"Saved {len(output_files)} file(s) to: {dest}")
for f in sorted(output_files):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:55s}  {size_kb:,.0f} KB")